# PHREEQC Surrogate Colab Training Control Panel

Goal: run the same training pipeline on Colab GPU while code stays in GitHub and large data stays in Google Drive.

Model contract for v1:

`Input.txt numeric conditions + time_d sequence -> full Output.txt trajectory`

Expected tensors:

- `X`: `(batch, 301, 20)` = 19 generic conditions plus normalized time
- `Y`: `(batch, 301, 32)` = PHREEQC output features
- Rock name is metadata only. It is not passed into the model input tensor.


In [26]:
import os
import shutil
import subprocess
from pathlib import Path

# Important: leave the repo folder before deleting it.
os.chdir("/content")

scheme = "https://"
host = "github" + ".com"
owner = "Hasan-ari"
repo = "chemical_thesis_repo"

REPO_URL = f"{scheme}{host}/{owner}/{repo}.git"
BRANCH = "main"
REPO_DIR = Path("/content/chemical_thesis_repo")

print("cwd before clone:", os.getcwd())
print("REPO_URL repr:", repr(REPO_URL))

shutil.rmtree(REPO_DIR, ignore_errors=True)

result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    cwd="/content",
    capture_output=True,
    text=True,
)

print("returncode:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("git clone failed")

os.chdir(REPO_DIR)

print("cwd after clone:", os.getcwd())

print("HEAD:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

print("Check data.py:")
subprocess.run(["ls", "-l", "conditional_model_v1/data.py"], check=True)

cwd before clone: /content
REPO_URL repr: 'https://github.com/Hasan-ari/chemical_thesis_repo.git'
returncode: 0
STDOUT:

STDERR:
Cloning into '/content/chemical_thesis_repo'...
Updating files:  94% (2780/2948)
Updating files:  95% (2801/2948)
Updating files:  96% (2831/2948)
Updating files:  97% (2860/2948)
Updating files:  98% (2890/2948)
Updating files:  99% (2919/2948)
Updating files: 100% (2948/2948)
Updating files: 100% (2948/2948), done.

cwd after clone: /content/chemical_thesis_repo
HEAD:
Check data.py:


CompletedProcess(args=['ls', '-l', 'conditional_model_v1/data.py'], returncode=0)

## 1. Settings

Edit these paths first. If the zip files are in **Shared with me**, add Drive shortcuts into **MyDrive** and point the paths below to those shortcuts.

In [21]:
from __future__ import annotations

from pathlib import Path

# Change this to your real GitHub repository URL after the code is pushed.
REPO_URL = "https://github.com/Hasan-ari/chemical_thesis_repo.git"
BRANCH = "main"
REPO_DIR = Path("/content/chemical_thesis_repo")

# Drive data location. Keep large zip files outside git.
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/chemical_thesis_data")
CALCITE_ZIP = DRIVE_DATA_DIR / "Calcite_wat_sat_data_3.zip"
DOLOMITE_ZIP = DRIVE_DATA_DIR / "Dolomite_wat_sat_data_2.zip"

# Fast Colab-local working folders.
DATA_ROOT = Path("/content/data")
PROCESSED_ROOT = REPO_DIR / "processed"
RUN_ROOT = REPO_DIR / "runs"
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/chemical_thesis_repo/runs")

RUN_SMOKE_FIRST = True
RUN_SWEEP = False

BASELINE_CONFIG = "configs/conditional_model_v1/full_colab_baseline.yaml"
SWEEP_CONFIGS = [
    "configs/conditional_model_v1/full_colab_lr1e-4.yaml",
    "configs/conditional_model_v1/full_colab_lr3e-4.yaml",
    "configs/conditional_model_v1/full_colab_baseline.yaml",
    "configs/conditional_model_v1/full_colab_lr3e-3.yaml",
    "configs/conditional_model_v1/full_colab_reduce_on_plateau.yaml",
    "configs/conditional_model_v1/full_colab_cosine.yaml",
]


## 2. Mount Drive

This gives Colab access to the zip files and to the output folder where run artifacts will be copied.

In [11]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Clone or Update the Repo

Colab pulls code from GitHub. After local changes are pushed, rerun this cell to get the latest code.

In [23]:
import os
import subprocess
import sys

if "YOUR_GITHUB_USER" in REPO_URL:
    raise ValueError("Set REPO_URL to your real GitHub repository before running this cell.")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")


CalledProcessError: Command '['git', 'clone', '--branch', 'main', 'https://github.com/Hasan-ari/chemical_thesis_repo.git', '/content/chemical_thesis_repo']' returned non-zero exit status 128.

## 4. Install Python Dependencies

This uses the repository `requirements.txt`. Colab already has many packages, but this makes the run reproducible.

In [13]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

## 5. Copy Data to the Colab VM

We train from `/content/data`, not directly from mounted Drive. This is faster because Drive is slow with many small files.

In [15]:
import shutil
import tempfile
import zipfile
from pathlib import Path

DATASETS = [
    (CALCITE_ZIP, DATA_ROOT / "wat_sat" / "Calcite", "Calcite_wat_sat_data_3"),
    (DOLOMITE_ZIP, DATA_ROOT / "wat_sat" / "Dolomite", "Dolomite_wat_sat_data_2"),
]

def find_dataset_root(root: Path) -> Path:
    """Find the folder that actually contains input/ and output/."""
    candidates = [root] + [path for path in root.rglob("*") if path.is_dir()]
    for path in candidates:
        if (path / "input").is_dir() and (path / "output").is_dir():
            return path
    raise FileNotFoundError(f"Could not find a dataset folder with input/ and output/ under {root}")

def extract_dataset(zip_path: Path, target_parent: Path, expected_folder: str) -> Path:
    """Extract zip and place dataset at the exact path expected by configs."""
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")

    expected_path = target_parent / expected_folder

    # Colab /content is temporary, so it is safe to rebuild this extracted copy.
    shutil.rmtree(expected_path, ignore_errors=True)
    target_parent.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_root = Path(tmp_dir)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(tmp_root)

        actual_dataset_root = find_dataset_root(tmp_root)
        shutil.copytree(actual_dataset_root, expected_path)

    print(f"Ready: {expected_path}")
    print(f"Input files example count: {len(list((expected_path / 'input').glob('*_Input.txt')))}")
    print(f"Output files example count: {len(list((expected_path / 'output').glob('*_Output.txt')))}")
    return expected_path

for zip_path, target_parent, expected_folder in DATASETS:
    extract_dataset(zip_path, target_parent, expected_folder)

Ready: /content/data/wat_sat/Calcite/Calcite_wat_sat_data_3
Input files example count: 9875
Output files example count: 9875
Ready: /content/data/wat_sat/Dolomite/Dolomite_wat_sat_data_2
Input files example count: 9882
Output files example count: 9882


## 6. Set Runtime Paths and Check GPU

The YAML configs use environment variables so private Drive paths are not committed to git.

In [16]:
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["PROCESSED_ROOT"] = str(PROCESSED_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)

RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.12.1+cu130
cuda available: True
gpu: Tesla T4


## 7. Smoke Run

This trains on 4 Calcite runs plus 4 Dolomite runs for 1 epoch. It proves the pipeline works before spending GPU time.

In [27]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "conditional_model_v1.cli.train",
        "--config",
        "configs/conditional_model_v1/smoke_colab.yaml",
    ],
    capture_output=True,
    text=True,
)

print("returncode:", result.returncode)
print("\nSTDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)

returncode: 0

STDOUT:
device=cuda
x_shape=(8, 301, 20) y_shape=(8, 301, 32)
split train=4 val=2 test=2
epoch=0001 train=1.041013e+00 val=2.799263e+00 lr=1.000e-03
run_dir=/content/chemical_thesis_repo/runs/20260629_005004_smoke_condition_lstm_colab


STDERR:



## 8. Full Training or Sweep

Start with the baseline. Set `RUN_SWEEP = True` in the settings cell when you want to compare learning rates and schedulers.

In [ ]:
configs_to_run = SWEEP_CONFIGS if RUN_SWEEP else [BASELINE_CONFIG]

for config_path in configs_to_run:
    print(f"Running {config_path}")
    subprocess.run(
        [sys.executable, "-m", "conditional_model_v1.cli.train", "--config", config_path],
        check=True,
    )


Running configs/conditional_model_v1/full_colab_baseline.yaml


## 9. Copy Run Artifacts Back to Drive Repo Folder

Every run folder contains config, history, metrics, checkpoints, plots, and full prediction arrays. Colab writes active outputs under the cloned repo at `/content/chemical_thesis_repo/runs`, then copies them to `MyDrive/chemical_thesis_repo/runs`.

In [ ]:
import shutil

DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_ROOT, DRIVE_RUN_ROOT, dirs_exist_ok=True)
print(f"Copied run artifacts to: {DRIVE_RUN_ROOT}")


## 10. Inspect Latest Run

`feature_metrics.csv` shows all output-feature errors. `eval_predictions.npz` stores every true and predicted trajectory for the eval split.

In [ ]:
import pandas as pd
from IPython.display import Image, display

run_dirs = sorted([path for path in RUN_ROOT.iterdir() if path.is_dir() and path.name[:8].isdigit()])
latest_run = run_dirs[-1]
print("latest run:", latest_run)

display(pd.read_csv(RUN_ROOT / "summary.csv").tail())
display(pd.read_csv(latest_run / "history.csv").tail())
display(pd.read_csv(latest_run / "feature_metrics.csv").sort_values("rmse_original", ascending=False).head(32))

grid_paths = sorted((latest_run / "plots" / "trajectory_examples").glob("*_all_outputs.png"))
for path in grid_paths[:3]:
    print(path.name)
    display(Image(filename=str(path)))

print("full eval arrays:", latest_run / "eval_predictions.npz")


## Key Meanings

- `history.csv`: loss after each epoch.
- `feature_metrics.csv`: one row per PHREEQC output feature.
- `eval_predictions.npz`: full numeric true/predicted trajectories.
- `*_all_outputs.png`: one visual grid showing all output trajectories for one run.
- `plots.max_runs: null`: render PNGs for every evaluated run. This can create many files on the full dataset.
